# 第99章 交叉验证与超参数调优

<!-- module-learning-arc:start -->
> **机器学习 模块主线｜第 14 / 34 步：从模型分数走向业务评价与阈值**
>
> **持续应用背景：** 建设可信预测系统：从统一训练流程开始，比较模型、处理不平衡、选择阈值、解释结果并保存完整 Pipeline，最终回答模型能否安全投入使用。
>
> **承接上一阶段：** 主成分分析（PCA）  →  **本章任务：** 交叉验证与超参数调优  →  **下一步：** 不平衡分类与阈值选择
>
> **大作业连接：** 本章练习将成为《模型上线评审会》的一部分，最终需要把候选模型变成经过预测合同、泄漏审计、业务阈值、错误分析和模型卡检查的上线建议。
<!-- module-learning-arc:end -->


## 本章场景

训练一个模型，最怕的是「看起来挺好、一上真实数据就翻车」——模型的参数总能在见过的那份数据上表现不错。


## 本章目标

学完本章，你将能够：

- **理解**：理解「交叉验证与超参数调优」的核心思想、适用场景、关键假设与要解释的业务问题。
- **操作**：能按标准流程完成数据准备、模型训练与评估，并解读「交叉验证与超参数调优」的关键输出指标。
- **迁移**：能把「交叉验证与超参数调优」迁移到一份新数据上，独立完成任务并就结果给出有分寸的结论。


## 核心概念

**背景引入**：训练一个模型，最怕的是「看起来挺好、一上真实数据就翻车」——模型的参数总能在见过的那份数据上表现不错。交叉验证把数据反复切分成多个折来反复检验，能更诚实地估计模型换个环境还能不能扛住；而超参数调优则是调整模型自己的「旋钮」（比如逻辑回归的惩罚强度 C），让它既不过度死记硬背、也不敷衍了事。掌握这两招，你才能判断一个模型是真的可靠，还是仅仅记性太好。

- 交叉验证重复利用训练数据估计泛化（打个比方：把同一批题反复拆成几套“模拟考”、轮流当正式考，取几次的平均——比只考一次更能看出真实水平，也更抗“碰巧考到背过的题”。）
- 分层折保持类别比例
- 超参数是拟合前配置而非模型学习参数
- 嵌套选择越多，越需要独立最终测试


## 方法分类速查

先用这张表建立本章的方法地图；每一行后面都有对应的独立示例或练习。

| 类别 | 常用方法或写法 | 主要用途 | 需要特别注意 |
| --- | --- | --- | --- |
| 多指标交叉验证 | `pd.DataFrame()`、`.agg()`、`.round()` | 比较逻辑回归在 5 个分层折上的准确率与 ROC-AUC 波动。 | 调参前已经多次查看测试集 |
| 流水线参数搜索 | `search.fit()`、`search.score()`、`pd.DataFrame()`、`.round()` | 参数名使用步骤名双下划线；搜索只使用训练集。 | 只报告最佳均值不报告波动 |


## 例 1｜多指标交叉验证

比较逻辑回归在 5 个分层折上的准确率与 ROC-AUC 波动。


<!-- math-foundation:chapter-99 -->
### 数学推导｜交叉验证汇总泛化波动

> 阅读方法：先跟着步骤理解每个量怎样产生，再看最后的可计算形式；不需要脱离业务场景死记公式。

**第 1 步｜把样本分成 $K$ 个互斥验证折。** 第 $k$ 次用其余折训练，在第 $k$ 折得到分数 $s_k$。

**第 2 步｜平均各折表现。** $\bar s=\sum_ks_k/K$ 近似描述该训练流程在不同样本划分下的表现。

**第 3 步｜报告划分敏感性。** 标准差衡量折间波动；若只想描述均值估计的不确定度，可另算

$$
SE(\bar s)\approx\frac{SD(s)}{\sqrt K}
$$

但各折训练集高度重叠，并不严格独立，所以这个标准误只能谨慎参考。

**把上面的关系收束为本章计算式：**

$$
\bar{s}=\frac{1}{K}\sum_{k=1}^{K}s_k,\qquad SD(s)=\sqrt{\frac{1}{K-1}\sum_k(s_k-\bar{s})^2}
$$

**符号解释：** $s_k$ 是第 $k$ 折验证分数。

**代码对应：** 同时报告均值和标准差，并选择与时间、分组或类别结构匹配的切分器。

**使用边界：** 交叉验证折并非完全独立；时间数据不能随机打乱未来与过去。


In [ ]:
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    cross_validate,
)
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

data = load_breast_cancer(as_frame=True)
X, y = data.data, data.target
X_train, X_test, y_train, y_test = train_test_split(
    X, y, stratify=y, test_size=0.2, random_state=88
)
pipe = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=88)
scores = cross_validate(
    pipe, X_train, y_train, cv=cv, scoring=["accuracy", "roc_auc"]
)
print(
    pd.DataFrame(scores)[["test_accuracy", "test_roc_auc"]]
    .agg(["mean", "std"])
    .round(4)
)


In [ ]:
# （自动维护）练习上下文快照 1：参考答案将基于此刻的变量运行
_pds_snap_1 = dict(globals())


**练一练**：交叉验证的结果会随折数 `n_splits` 和切分随机种子而变。请把「示例 1」里的折数从 5 改成 3，再次计算每个折的准确率与 ROC-AUC，观察均值与标准差的变化，并把最明显的差异写进观察记录。提示：直接复用前面已经定义好的 `X_train` / `y_train` / `pipe`，只构造一个新的折数为 3 的交叉验证器即可。


In [ ]:
try:
    # 请在下方填写代码：把交叉验证的折数 n_splits 从 5 改成 3，再计算每个折的准确率与 ROC-AUC。
    # 提示：可复用前面已经定义好的 X_train / y_train / pipe。
    import pandas as pd
    from sklearn.model_selection import StratifiedKFold, cross_validate

    # ---- 填写区：构造折数为 3 的交叉验证器（把 n_splits 改成 3）----
    # ---- 填写区：用 cross_validate 计算 accuracy 与 roc_auc ----

except Exception as _pds_err:
    print("（练习尚未完成或未填全：", _pds_err, "）")


## 例 2｜流水线参数搜索

参数名使用步骤名双下划线；搜索只使用训练集。


In [ ]:
from sklearn.model_selection import GridSearchCV

search = GridSearchCV(
    pipe,
    {"logisticregression__C": [0.01, 0.1, 1, 10, 100]},
    scoring="roc_auc",
    cv=cv,
    n_jobs=-1,
    return_train_score=True,
)
search.fit(X_train, y_train)
print(
    "最佳参数:", search.best_params_, " CV AUC:", round(search.best_score_, 4)
)
print("独立测试 AUC:", round(search.score(X_test, y_test), 4))
display(
    pd.DataFrame(search.cv_results_)[
        [
            "param_logisticregression__C",
            "mean_train_score",
            "mean_test_score",
            "std_test_score",
        ]
    ].round(4)
)


## 独立迁移练习

在不改变数据切分和指标的前提下，比较基线与一个模型设置。

先在下面单元格完成自己的版本；需要参考时再回看紧邻的示例或参考实现。


In [ ]:
# （自动维护）练习上下文快照 2：参考答案将基于此刻的变量运行
_pds_snap_2 = dict(globals())


In [ ]:
try:
    # TODO: 在此粘贴或改写最接近的示例。
    # 记录：我改了什么？预期会发生什么？实际观察到什么？
    change_note = "待填写"
    expected_change = "待填写"
    observed_change = "运行后填写"
    print(
        {"修改": change_note, "预期": expected_change, "观察": observed_change}
    )

except Exception as _pds_err:
    print("（练习尚未完成或未填全：", _pds_err, "）")


## 本章实训：模型与基线比较

这一组实验专门训练“观察一个结果 → 只改一个变量 → 解释变化”。先运行第一个代码单元格，再运行第二个。


In [ ]:
import numpy as np
import pandas as pd
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error

X = pd.DataFrame(
    {"visits": [1, 2, 3, 4, 5, 6], "discount": [0, 0, 1, 1, 1, 2]}
)
y = np.array([12, 15, 19, 23, 27, 31])
baseline = DummyRegressor(strategy="mean").fit(X, y)
model = LinearRegression().fit(X, y)
print("基线预测：", np.round(baseline.predict(X[:2]), 2))
print("模型预测：", np.round(model.predict(X[:2]), 2))
print("基线MAE：", round(mean_absolute_error(y, baseline.predict(X)), 2))
print("模型MAE：", round(mean_absolute_error(y, model.predict(X)), 2))


### 第一个结果怎么读

复杂模型之前先建立基线。只有在同一数据切分和同一指标下超过基线，模型才值得继续分析。

请记录：输入是什么、输出是什么、输出支持了哪一个结论。


In [ ]:
X_changed = X.copy()
X_changed["visits"] = X_changed["visits"] + 1
changed_prediction = model.predict(X_changed)
print("原始前2个预测：", np.round(model.predict(X[:2]), 2))
print("访问次数+1后的预测：", np.round(changed_prediction[:2], 2))
print("预测变化：", np.round(changed_prediction[:2] - model.predict(X[:2]), 2))


### 第二个结果怎么读

只把一个特征整体加 1，观察预测变化。这个实验只能说明模型的预测响应，不能直接证明真实世界的因果关系。

迁移任务：把一个输入值、一个字段或一个图表参数换成自己的例子，再用一句话解释变化。


## 错误恢复：模型特征泄漏怎么办

真实数据和真实代码都会出问题。本节先观察问题，再用一个明确的检查或修复步骤恢复运行。


In [ ]:
import pandas as pd

_demo_data = pd.DataFrame(
    {
        "visits": [2, 4, 6],
        "duration_after_call": [30, 80, 120],
        "target": [0, 1, 1],
    }
)
forbidden = {"target", "duration_after_call"}
features = [column for column in _demo_data.columns if column not in forbidden]
print("禁止使用：", sorted(forbidden))
print("安全特征：", features)
print("原因：特征必须在预测时点已经可获得。")


### 错误恢复步骤

1. 先看错误类型、字段或数据形状。
2. 判断问题发生在输入、处理中间结果还是输出。
3. 修复后重新检查结果，而不是只让代码不报错。

如果一个字段在结果发生之后才产生，它即使与目标高度相关，也不能作为预测特征。先定义预测时点，再列可用字段。

迁移任务：把示例中的输入换成一组会触发问题的数据，并记录你的修复规则。


## 易错点提醒

- 调参前已经多次查看测试集
- 只报告最佳均值不报告波动
- 预处理不在 Pipeline 内导致折间泄漏
- 盲目扩大搜索空间造成多重尝试偏差


## 练习与作业

1. 同时搜索 penalty='l1'/'l2' 和 C
2. 使用 solver='liblinear'
3. 输出最优参数和测试 AUC

提交前检查：代码可从上到下运行，关键中间结果可核对，结论注明计算口径。

## 99.11 练习路径

1. **跟练**：先运行示例，确认输出结构，再完成“同时搜索 penalty='l1'/'l2' 和 C”。
2. **独立完成**：不复制示例代码，完成“使用 solver='liblinear'”，并保留一个中间结果用于检查。
3. **迁移挑战**：尝试“输出最优参数和测试 AUC”，用一两句话说明你修改了什么。

### 99.11.1 完成标准

- 代码从上到下运行不报错，关键变量类型和形状符合预期。
- 至少输出一个可核对的数值、表格或图形，并写明计算口径。
- 结论能够回答任务问题，同时说明一个限制或未验证的假设。

### 99.11.2 分级提示

- **提示 1**：先复用示例中的数据结构和变量命名。
- **提示 2**：把任务拆成“准备数据 → 计算 → 检查 → 表达”四步。
- **提示 3**：运行隐藏答案前，先用 type()、shape、head() 或断言定位问题。


## 小结

使用交叉验证估计模型波动，并用 GridSearchCV 在训练数据内部选择超参数，最后只在独立测试集评估一次。


### 你已经掌握

- 使用 StratifiedKFold
- 同时报告均值和标准差
- 用 GridSearchCV 搜索流水线参数
- 保持最终测试集独立


### 需要注意

- 调参前已经多次查看测试集
- 只报告最佳均值不报告波动
- 预处理不在 Pipeline 内导致折间泄漏
- 盲目扩大搜索空间造成多重尝试偏差


## 参考答案


### 本章练习


In [ ]:
# 恢复练习 1 时的变量上下文（后面的示例覆盖过这些名字）
globals().update(_pds_snap_1)


In [ ]:
# 完整答案：折数改为 3，其余保持一致。
import pandas as pd
from sklearn.model_selection import StratifiedKFold, cross_validate

cv_3 = StratifiedKFold(n_splits=3, shuffle=True, random_state=88)
scores_3 = cross_validate(
    pipe, X_train, y_train, cv=cv_3, scoring=["accuracy", "roc_auc"]
)
result_3 = (
    pd.DataFrame(scores_3)[["test_accuracy", "test_roc_auc"]]
    .agg(["mean", "std"])
    .round(4)
)
print(result_3)


### 本章练习


In [ ]:
# 恢复练习 2 时的变量上下文（后面的示例覆盖过这些名字）
globals().update(_pds_snap_2)


In [ ]:
practice_pipe = make_pipeline(
    StandardScaler(), LogisticRegression(max_iter=1000, solver="liblinear")
)
practice_search = GridSearchCV(
    practice_pipe,
    {
        "logisticregression__C": [0.01, 0.1, 1, 10],
        "logisticregression__penalty": ["l1", "l2"],
    },
    scoring="roc_auc",
    cv=cv,
    n_jobs=-1,
)
practice_search.fit(X_train, y_train)
practice_test_score = practice_search.score(X_test, y_test)
print(practice_search.best_params_, round(practice_test_score, 4))
